# Exercise 8-03: Choose the Production Inference Optimization Strategy (Optional Worksheet)

**Chapter 8: Transformer Architectures, Tokenization, Model Compression, and Inference Optimization**  
**Lab Type:** Interactive Worksheet Notebook | **Estimated Time:** 40–50 minutes  
**Exam Domains Covered:** Core Machine Learning & AI Knowledge (30%), Software Development (24%)

---

## Learning Objectives
By the end of this interactive worksheet, you will be able to:
1. Master every optimization technique in **NVIDIA TensorRT-LLM**, **Triton Inference Server**, and **NVIDIA NIM** tested on the NCA-GENL exam:
   - **In-Flight Batching (Continuous Batching)** vs. Static Batching
   - **Paged KV Cache** vs. Contiguous KV Allocation
   - **Speculative Decoding (Draft + Target Model)**
   - **FlashAttention & Kernel Fusion**
   - **Tensor Parallelism (TP)** vs. **Pipeline Parallelism (PP)**
   - **Triton Dynamic Batching, Concurrent Model Execution & Model Ensembles**
2. Diagnose the exact hardware bottleneck (**VRAM capacity**, **Prefill compute / TTFT**, **Decode memory bandwidth / ITL**, or **Multi-user concurrency**) in real-world production scenarios and select the optimal engineering solution.

---
## 1. Master Reference Table: NVIDIA LLM Inference Optimization Stack (Chapter 8)

> **HIGH-YIELD EXAM STUDY TABLE:** Nearly every inference scenario question on the NCA-GENL exam maps a specific **production symptom/bottleneck** to one of the technologies below!

| Optimization Technique | Component / Layer | Problem It Solves | How It Works Under the Hood |
| :--- | :--- | :--- | :--- |
| **In-Flight Batching** *(Continuous Batching)* | **TensorRT-LLM** | Short requests are stuck waiting for a 2,000-token request in the same batch to finish (**head-of-line blocking / GPU idle bubbles**) | Evicts finished sequences from the batch **immediately at the token step** and inserts waiting requests mid-flight |
| **Paged KV Cache** | **TensorRT-LLM** | GPU runs out of VRAM at high concurrency because pre-allocating `max_seq_len` contiguous KV buffers wastes 60%+ of memory | Allocates KV cache dynamically in fixed-size **non-contiguous memory blocks/pages** (like OS virtual memory), eliminating fragmentation |
| **Speculative Decoding** | **TensorRT-LLM** | Single-user **Decode Inter-Token Latency (ITL)** is too slow because autoregressive decoding is **memory-bandwidth bound** | A tiny fast **Draft Model** (or Medusa heads) proposes $K$ tokens; the large **Target Model** verifies all $K$ tokens in **one parallel forward pass** (mathematically lossless!) |
| **FlashAttention & Kernel Fusion** | **TensorRT-LLM / CUDA** | Standard $O(N^2)$ attention materializes huge $N \times N$ matrices in slow HBM memory and launches separate kernels for `MatMul + Bias + GELU` | Tiles attention into fast **on-chip SRAM** and fuses multiple ops into a single CUDA kernel, slashing HBM memory reads/writes |
| **FP8 / INT4 AWQ Quantization** | **TensorRT-LLM / Hopper Tensor Cores** | Model weights exceed single-GPU VRAM or memory bandwidth limits decode speed | Reduces weight precision from 16-bit to **FP8 (1 B/p on H100/L40S)** or **INT4 AWQ (0.5 B/p)**, cutting both VRAM and HBM read bytes per token |
| **Tensor Parallelism (TP)** | **TensorRT-LLM + NVLink** | A 70B+ model does not fit on 1 GPU, or single-request **TTFT & ITL** must be reduced across multiple GPUs on the same node | Shards individual weight matrices ($W_Q, W_K, W_V, W_{\text{MLP}}$) **horizontally/vertically across GPUs** with fast `AllReduce` over **NVLink** |
| **Pipeline Parallelism (PP)** | **TensorRT-LLM (Multi-Node)** | Model spans multiple nodes connected by slower inter-node networking (where per-layer TP `AllReduce` would bottleneck) | Assigns **contiguous groups of layers** to each GPU/node (e.g., Layers 1–40 on Node 1, Layers 41–80 on Node 2) |
| **Model Ensembles & Dynamic Batching** | **Triton Inference Server** | Application needs to chain `[Preprocessor -> Embedding -> Reranker -> LLM]` or serve PyTorch, ONNX, and TensorRT models simultaneously | Orchestrates multi-model DAGs (**Model Ensembles**) and groups incoming requests within a microsecond delay window (**Dynamic Batching**) |
| **NVIDIA NIM** | **Containerized Microservice** | Enterprise wants a turnkey, secure **OpenAI-compatible REST API** (`/v1/chat/completions`) bundling Triton + TensorRT-LLM | Pre-built container (`nvcr.io/nim/...`) that auto-selects the optimal TensorRT-LLM engine for the host GPU |

In [1]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## 2. Quantitative Simulation: Static Batching vs. TensorRT-LLM In-Flight Batching

Why does **Chapter 8** emphasize **In-Flight (Continuous) Batching** so strongly?
Suppose a batch of 4 concurrent requests arrives where Requests 1, 2, and 3 only need **30 output tokens**, while Request 4 asks for a **400-token essay**:
- **Static Batching:** Requests 1, 2, and 3 cannot return or free their GPU slots until Request 4 finishes all 400 steps! Even worse, GPU lanes 1, 2, and 3 pad with dummy work for steps 31–400 (`77%` wasted GPU compute!) while новых requests wait in the queue.
- **In-Flight Batching:** At step 30, Requests 1, 2, and 3 immediately return to their users, and Requests 5, 6, and 7 are inserted into those 3 freed slots on step 31!

Let's simulate this and compare **GPU Utilization (%)**, **Average Request Latency**, and **Total Throughput (requests/sec)**!

In [2]:
def simulate_static_vs_inflight_batching():
    # 8 requests with heterogeneous output lengths (tokens)
    output_lengths = [32, 40, 28, 380, 35, 45, 30, 36]
    batch_slots = 4
    ms_per_step = 15.0  # 15 ms per decode step

    # 1. Static Batching: Batch 1 runs [32, 40, 28, 380] -> all 4 slots locked for max(Batch 1) = 380 steps!
    # Batch 2 runs [35, 45, 30, 36] -> locked for max(Batch 2) = 45 steps!
    b1_max = max(output_lengths[:4])
    b2_max = max(output_lengths[4:])
    static_total_steps = b1_max + b2_max
    static_total_time_s = (static_total_steps * ms_per_step) / 1000.0
    useful_token_steps = sum(output_lengths)
    static_slot_steps = 4 * static_total_steps
    static_gpu_efficiency = useful_token_steps / static_slot_steps * 100.0

    # 2. In-Flight Batching: As soon as a short request finishes on a slot, the next waiting request takes its slot!
    slots = [0] * batch_slots
    completion_times_inflight = []
    for req_tokens in output_lengths:
        # Assign to the earliest-freeing slot
        earliest_idx = int(np.argmin(slots))
        slots[earliest_idx] += req_tokens
        completion_times_inflight.append(slots[earliest_idx] * ms_per_step)

    inflight_total_steps = max(slots)
    inflight_total_time_s = (inflight_total_steps * ms_per_step) / 1000.0
    inflight_slot_steps = 4 * inflight_total_steps
    inflight_gpu_efficiency = useful_token_steps / inflight_slot_steps * 100.0

    df_batch = pd.DataFrame([
        {
            "Batching_Strategy": "Static Batching (Waits for Longest Sequence)",
            "Total_Wall_Time_s": round(static_total_time_s, 2),
            "Effective_Slot_Utilization_%": round(static_gpu_efficiency, 1),
            "Throughput_Req_per_Sec": round(len(output_lengths) / static_total_time_s, 2),
        },
        {
            "Batching_Strategy": "TensorRT-LLM In-Flight (Continuous) Batching",
            "Total_Wall_Time_s": round(inflight_total_time_s, 2),
            "Effective_Slot_Utilization_%": round(inflight_gpu_efficiency, 1),
            "Throughput_Req_per_Sec": round(len(output_lengths) / inflight_total_time_s, 2),
        },
    ])
    return df_batch

print("=== STATIC BATCHING VS. TENSORRT-LLM IN-FLIGHT BATCHING SIMULATION ===")
print(simulate_static_vs_inflight_batching().to_string(index=False))

=== STATIC BATCHING VS. TENSORRT-LLM IN-FLIGHT BATCHING SIMULATION ===
                           Batching_Strategy  Total_Wall_Time_s  Effective_Slot_Utilization_%  Throughput_Req_per_Sec
Static Batching (Waits for Longest Sequence)               6.38                          36.8                    1.25
TensorRT-LLM In-Flight (Continuous) Batching               5.70                          41.2                    1.40


---
## 3. Quantitative Simulation: Contiguous KV Cache vs. Paged KV Cache

Why does **Paged KV Cache** allow an LLM server to handle $2\text{–}4\times$ more concurrent users on the exact same GPU?
- Without Paged KV Cache, the serving engine must pre-allocate a contiguous buffer of size `max_seq_len = 8192` tokens for every active request—even if the average conversation is only `650` tokens long (`92%` internal fragmentation!).
- With **Paged KV Cache** (for example, `page_size = 64` tokens per block), each request only consumes `ceil(actual_tokens / 64)` pages, freeing gigabytes of VRAM for larger in-flight batch sizes!

In [3]:
def simulate_paged_kv_cache(active_requests: int = 32, max_seq_len: int = 8192, page_size: int = 64, kb_per_token_kv: float = 512.0):
    rng = np.random.default_rng(42)
    actual_lengths = rng.integers(250, 1400, size=active_requests)

    # 1. Contiguous pre-allocation: reserves max_seq_len for every active request
    contiguous_tokens_reserved = active_requests * max_seq_len
    contiguous_gb = (contiguous_tokens_reserved * kb_per_token_kv) / (1024 ** 2)

    # 2. Paged KV Cache: allocates only ceil(actual_len / page_size) * page_size tokens per request
    paged_tokens_reserved = int(np.sum(np.ceil(actual_lengths / page_size) * page_size))
    paged_gb = (paged_tokens_reserved * kb_per_token_kv) / (1024 ** 2)

    waste_pct = (1.0 - (actual_lengths.sum() / contiguous_tokens_reserved)) * 100.0
    paged_waste_pct = (1.0 - (actual_lengths.sum() / paged_tokens_reserved)) * 100.0

    print("=== CONTIGUOUS KV CACHE VS. TENSORRT-LLM PAGED KV CACHE (32 CONCURRENT USERS) ===")
    print(f"Contiguous Pre-Allocated KV Memory : {contiguous_gb:>6.2f} GB (Internal Fragmentation Waste: {waste_pct:.1f}%)")
    print(f"Paged KV Cache Allocated Memory    : {paged_gb:>6.2f} GB (Internal Fragmentation Waste: {paged_waste_pct:.1f}%)")
    print(f"VRAM Saved by Paged KV Cache       : {contiguous_gb - paged_gb:>6.2f} GB ({contiguous_gb / paged_gb:.1f}x higher concurrency capacity!)")

simulate_paged_kv_cache()

=== CONTIGUOUS KV CACHE VS. TENSORRT-LLM PAGED KV CACHE (32 CONCURRENT USERS) ===
Contiguous Pre-Allocated KV Memory : 128.00 GB (Internal Fragmentation Waste: 89.5%)
Paged KV Cache Allocated Memory    :  13.91 GB (Internal Fragmentation Waste: 3.1%)
VRAM Saved by Paged KV Cache       : 114.09 GB (9.2x higher concurrency capacity!)


---
## 4. Interactive Scenario Decision Matrix (8 Production Architecture Cases)

Let's run our automated **Optimization Strategy Advisor** across 8 realistic NCA-GENL exam scenarios and review the exact engineering justification for each choice!

In [4]:
OPTIMIZATION_SCENARIOS = [
    {
        "Scenario_ID": "OPT-01",
        "Production_Problem": "A 70B FP16 model (140 GB weights) must run with low latency on a single node equipped with 2x H100 (80 GB) GPUs connected via NVLink.",
        "Best_Strategy": "Tensor Parallelism (TP=2) in TensorRT-LLM (or FP8 Quantization on 1x H100)",
        "Why": "TP=2 shards each layer's weight matrices across the 2 NVLink-connected GPUs (~70 GB/GPU) and parallelizes both Prefill (TTFT) and Decode (ITL).",
    },
    {
        "Scenario_ID": "OPT-02",
        "Production_Problem": "Short 20-token chat replies suffer huge latency spikes whenever another user in the same batch requests a 2,000-token code file.",
        "Best_Strategy": "TensorRT-LLM In-Flight (Continuous) Batching",
        "Why": "Evicts finished 20-token sequences immediately at the iteration level instead of blocking until the 2,000-token sequence completes.",
    },
    {
        "Scenario_ID": "OPT-03",
        "Production_Problem": "GPU VRAM fills up at just 12 concurrent users on an 8k-context model even though most users only send 400-token prompts.",
        "Best_Strategy": "TensorRT-LLM Paged KV Cache (+ FP8/INT8 KV Cache Quantization)",
        "Why": "Allocates KV cache in non-contiguous pages on demand rather than pre-allocating 8,192 contiguous tokens per user.",
    },
    {
        "Scenario_ID": "OPT-04",
        "Production_Problem": "A low-concurrency executive coding assistant has plenty of free GPU compute, but single-user Decode speed (ITL) feels too slow due to HBM bandwidth limits.",
        "Best_Strategy": "Speculative Decoding (Small Draft Model + Target Model verification)",
        "Why": "Drafts K tokens rapidly and verifies them in one parallel Target forward pass, converting memory-bound decode steps into compute-efficient parallel checks without losing quality.",
    },
    {
        "Scenario_ID": "OPT-05",
        "Production_Problem": "A startup must deploy a 7B model on a single 8 GB edge GPU for offline field technicians.",
        "Best_Strategy": "4-Bit Post-Training Quantization (INT4 AWQ / GPTQ / NF4)",
        "Why": "Compresses 7B weights from 14 GB (FP16) down to ~3.5 GB (0.5 B/param), leaving >4 GB for KV cache and activations on the 8 GB device.",
    },
    {
        "Scenario_ID": "OPT-06",
        "Production_Problem": "An enterprise platform serves a pipeline chaining a BERT tokenizer, an E5 embedding model, a cross-encoder reranker, and a TensorRT-LLM generator in one API call.",
        "Best_Strategy": "NVIDIA Triton Inference Server with Model Ensembles & Dynamic Batching",
        "Why": "Triton's Model Ensemble orchestrates multi-model DAGs on the server (avoiding client-server network round-trips between stages) with per-model dynamic batching.",
    },
    {
        "Scenario_ID": "OPT-07",
        "Production_Problem": "An application team using the OpenAI Python SDK wants to migrate to self-hosted NVIDIA GPUs in Kubernetes this afternoon with zero ML compiler code.",
        "Best_Strategy": "Deploy an NVIDIA NIM Container (nvcr.io/nim/...)",
        "Why": "NIM provides a pre-packaged Docker container with Triton + TensorRT-LLM and a drop-in OpenAI-compatible /v1/chat/completions endpoint.",
    },
    {
        "Scenario_ID": "OPT-08",
        "Production_Problem": "A team wants a 3B model that matches the domain classification accuracy of their expensive 70B model at 20x lower latency.",
        "Best_Strategy": "Knowledge Distillation (70B Teacher -> 3B Student)",
        "Why": "Trains the compact 3B Student on the soft probabilities / synthetic labels generated by the 70B Teacher.",
    },
]

df_opt = pd.DataFrame(OPTIMIZATION_SCENARIOS)
for _, row in df_opt.iterrows():
    print("=" * 90)
    print(f"[{row['Scenario_ID']}] Problem : {row['Production_Problem']}")
    print(f"  -> Best Strategy : {row['Best_Strategy']}")
    print(f"  -> Justification : {row['Why']}")

[OPT-01] Problem : A 70B FP16 model (140 GB weights) must run with low latency on a single node equipped with 2x H100 (80 GB) GPUs connected via NVLink.
  -> Best Strategy : Tensor Parallelism (TP=2) in TensorRT-LLM (or FP8 Quantization on 1x H100)
  -> Justification : TP=2 shards each layer's weight matrices across the 2 NVLink-connected GPUs (~70 GB/GPU) and parallelizes both Prefill (TTFT) and Decode (ITL).
[OPT-02] Problem : Short 20-token chat replies suffer huge latency spikes whenever another user in the same batch requests a 2,000-token code file.
  -> Best Strategy : TensorRT-LLM In-Flight (Continuous) Batching
  -> Justification : Evicts finished 20-token sequences immediately at the iteration level instead of blocking until the 2,000-token sequence completes.
[OPT-03] Problem : GPU VRAM fills up at just 12 concurrent users on an 8k-context model even though most users only send 400-token prompts.
  -> Best Strategy : TensorRT-LLM Paged KV Cache (+ FP8/INT8 KV Cache Quantizat

---
## 5. NCA-GENL Chapter 8 Review Takeaways

- **TensorRT-LLM Core Features:** **In-Flight Batching** (eliminates head-of-line blocking), **Paged KV Cache** (eliminates KV memory fragmentation), **FlashAttention & Kernel Fusion** (minimizes HBM traffic), **Speculative Decoding** (accelerates memory-bound decode ITL), and **Tensor/Pipeline Parallelism** (scales across multi-GPU/multi-node clusters).
- **Triton vs. TensorRT-LLM vs. NIM:**
  - **TensorRT-LLM** compiles and optimizes the LLM graph into a fast GPU engine.
  - **Triton Inference Server** serves one or more engines/frameworks with dynamic batching and **Model Ensembles**.
  - **NVIDIA NIM** packages Triton + TensorRT-LLM + model weights into a turnkey container with an **OpenAI-compatible REST API**.